# Práctica · Semana 3: Búsqueda informada, búsqueda local y metaheurísticas

**Curso:** Introducción a la Inteligencia Artificial · UDES
**Tiempo estimado:** 2 h 30 min
**Relación con el capítulo:** secciones 3.1 a 3.9

En la semana 2 resolviste problemas «a ciegas». Ahora vas a darle a la búsqueda **información sobre el problema** y a comprobar cuánto trabajo ahorra. El cuaderno tiene dos mitades:

| Parte | Problema | Técnicas |
|---|---|---|
| **Búsqueda de caminos** | Las rutas por carretera de la semana 2 y el 8-puzle | Voraz, A\*, heurísticas admisibles y no admisibles, vuelta atrás con las *N* reinas |
| **Optimización** | Un viajante de comercio (TSP) por 24 capitales de Colombia | Vecino más cercano, *hill climbing*, temple simulado, búsqueda tabú y algoritmo genético |

Ejecuta las celdas en orden. Las celdas marcadas con ✏️ son ejercicios: responde en la celda de texto que las sigue. Todas las celdas se probaron con Python 3 y solo usan librerías que Google Colab trae instaladas.

In [ ]:
import heapq
import math
import random
import time
from collections import deque

import matplotlib.pyplot as plt

---
# Parte I. Búsqueda de caminos

## 1. El mapa de carreteras con coordenadas

Recuperamos la red de carreteras de García Serrano (2016, cap. 3) que usaste en la semana 2. Para dar información a la búsqueda añadimos las **coordenadas geográficas** de cada ciudad (latitud y longitud aproximadas), con las que calculamos la **distancia en línea recta** entre dos ciudades mediante la fórmula del haversine.

Esa distancia es la heurística $h_{LR}(n)$ de la sección 3.1: una estimación de lo que falta para llegar al destino. Como ninguna carretera puede ser más corta que la línea recta, $h_{LR}$ **nunca sobreestima**: es admisible. La celda lo comprueba en cada tramo.

In [ ]:
carreteras = {
    "Malaga":    {"Granada": 125, "Madrid": 513},
    "Sevilla":   {"Madrid": 514},
    "Granada":   {"Malaga": 125, "Madrid": 423, "Valencia": 491},
    "Valencia":  {"Granada": 491, "Madrid": 356, "Zaragoza": 309, "Barcelona": 346},
    "Madrid":    {"Salamanca": 203, "Sevilla": 514, "Malaga": 513, "Granada": 423,
                  "Barcelona": 603, "Santander": 437, "Valencia": 356,
                  "Zaragoza": 313, "Santiago": 599},
    "Salamanca": {"Santiago": 390, "Madrid": 203},
    "Santiago":  {"Salamanca": 390, "Madrid": 599},
    "Santander": {"Madrid": 437, "Zaragoza": 394},
    "Zaragoza":  {"Barcelona": 296, "Valencia": 309, "Madrid": 313},
    "Barcelona": {"Zaragoza": 296, "Madrid": 603, "Valencia": 346},
}

# Latitud y longitud aproximadas del centro de cada ciudad
coordenadas = {
    "Malaga": (36.7213, -4.4214),    "Sevilla": (37.3891, -5.9845),
    "Granada": (37.1773, -3.5986),   "Valencia": (39.4699, -0.3763),
    "Madrid": (40.4168, -3.7038),    "Salamanca": (40.9701, -5.6635),
    "Santiago": (42.8782, -8.5448),  "Santander": (43.4623, -3.8100),
    "Zaragoza": (41.6488, -0.8891),  "Barcelona": (41.3874, 2.1686),
}


def linea_recta(a, b, coords=coordenadas):
    # Distancia del círculo máximo (haversine) en kilómetros.
    (la1, lo1), (la2, lo2) = coords[a], coords[b]
    la1, lo1, la2, lo2 = map(math.radians, (la1, lo1, la2, lo2))
    h = math.sin((la2 - la1) / 2) ** 2 + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371 * math.asin(math.sqrt(h))


for a in carreteras:
    for b, km in carreteras[a].items():
        assert km >= linea_recta(a, b), (a, b)
print("La línea recta nunca supera la distancia por carretera: h_LR es admisible.")
print("h_LR(·, Santiago):", {c: round(linea_recta(c, "Santiago")) for c in coordenadas})

## 2. Una sola función para tres estrategias

La búsqueda de coste uniforme, la voraz y A\* son el **mismo algoritmo** con distinta función de evaluación $f(n)$ (sección 3.3):

| Estrategia | $f(n)$ | Qué tiene en cuenta |
|---|---|---|
| Coste uniforme | $g(n)$ | Solo lo recorrido |
| Voraz primero el mejor | $h(n)$ | Solo lo que falta (estimado) |
| A\* | $g(n) + h(n)$ | Las dos cosas |

La función `primero_el_mejor` recibe $f$ como parámetro. Con `traza=True` muestra, en cada paso, el nodo seleccionado con sus valores $g$, $h$ y $f$.

In [ ]:
class Nodo:
    def __init__(self, estado, padre=None, g=0):
        self.estado, self.padre, self.g = estado, padre, g
        self.profundidad = 0 if padre is None else padre.profundidad + 1

    def camino(self):
        n, c = self, []
        while n:
            c.append(n.estado)
            n = n.padre
        return c[::-1]


def primero_el_mejor(inicial, es_objetivo, sucesores, f, traza=False, h=None):
    # Búsqueda en grafos primero el mejor. sucesores(e) -> [(hijo, costo)]; f(nodo) -> prioridad.
    raiz = Nodo(inicial)
    contador = 0                                   # desempate estable
    frontera = [(f(raiz), contador, raiz)]
    mejor_g = {inicial: 0}
    cerrados = set()
    expandidos = 0
    while frontera:
        fn, _, nodo = heapq.heappop(frontera)
        if nodo.estado in cerrados:
            continue                               # entrada obsoleta
        if traza:
            hn = h(nodo) if h else 0
            print(f"  expande {nodo.estado:<10} g={nodo.g:>5.0f}  h={hn:>5.0f}  f={fn:>5.0f}")
        if es_objetivo(nodo.estado):
            return nodo, expandidos
        cerrados.add(nodo.estado)
        expandidos += 1
        for hijo, costo in sucesores(nodo.estado):
            g = nodo.g + costo
            if hijo not in cerrados and g < mejor_g.get(hijo, float("inf")):
                mejor_g[hijo] = g
                contador += 1
                h_nodo = Nodo(hijo, nodo, g)
                heapq.heappush(frontera, (f(h_nodo), contador, h_nodo))
    return None, expandidos


def resolver_ruta(origen, destino, estrategia, traza=False, factor=1.0):
    sucesores = lambda c: sorted(carreteras[c].items())
    h = lambda n: factor * linea_recta(n.estado, destino)
    f = {"coste uniforme": lambda n: n.g,
         "voraz": h,
         "A*": lambda n: n.g + h(n)}[estrategia]
    if traza:
        print(f"{estrategia}: {origen} -> {destino}")
    return primero_el_mejor(origen, lambda c: c == destino, sucesores, f, traza, h)


for estrategia in ["coste uniforme", "voraz", "A*"]:
    nodo, exp = resolver_ruta("Malaga", "Santiago", estrategia)
    print(f"{estrategia:<15} {' -> '.join(nodo.camino()):<45} {nodo.g:>5} km   expandidos: {exp}")

Veamos las trazas paso a paso de A\* y de la búsqueda voraz. En cada línea aparece el nodo que sale de la frontera para ser expandido, con sus valores de $g$, $h$ y $f$.

In [ ]:
resolver_ruta("Malaga", "Santiago", "A*", traza=True)
print()
resolver_ruta("Malaga", "Santiago", "voraz", traza=True);

### ✏️ Ejercicio 1. ¿Cuánto ahorra la heurística?

La celda siguiente resuelve los 90 pares origen-destino posibles con las tres estrategias.

1. ¿En cuántos casos la búsqueda **voraz** devuelve una ruta más larga que la óptima? Escribe un ejemplo y explica, con la traza, por qué se equivoca.
2. ¿A\* encontró siempre la ruta óptima? ¿Qué propiedad de $h_{LR}$ lo garantiza (sección 3.3.3)?
3. Compara el promedio de nodos expandidos por coste uniforme y por A\*. ¿Por qué el ahorro es modesto en este mapa? Piensa en cuántas ciudades tiene y en cuánto se parece la línea recta a la carretera.

In [ ]:
ciudades = list(carreteras)
totales = {"coste uniforme": 0, "voraz": 0, "A*": 0}
no_optimas = {"voraz": [], "A*": []}
for o in ciudades:
    for d in ciudades:
        if o == d:
            continue
        optimo, e = resolver_ruta(o, d, "coste uniforme")
        totales["coste uniforme"] += e
        for est in ["voraz", "A*"]:
            nodo, e = resolver_ruta(o, d, est)
            totales[est] += e
            if nodo.g > optimo.g:
                no_optimas[est].append((o, d, nodo.g, optimo.g))

n = len(ciudades) * (len(ciudades) - 1)
for est, t in totales.items():
    print(f"{est:<15} nodos expandidos en promedio: {t / n:.2f}")
for est, casos in no_optimas.items():
    print(f"\n{est}: {len(casos)} rutas no óptimas de {n}")
    for o, d, g, opt in casos[:5]:
        print(f"   {o} -> {d}: {g} km (óptimo {opt} km)")

*Tus respuestas al ejercicio 1:*

### ✏️ Ejercicio 2. Cuando la heurística exagera

Multiplicar una heurística admisible por un factor mayor que 1 la vuelve **no admisible**: puede sobreestimar lo que falta. Esto se usa en la práctica para buscar más rápido a cambio de perder la garantía de optimalidad (sección 3.3.4).

1. Ejecuta la celda con factores 1, 1,5, 2 y 4. ¿Qué pasa con los nodos expandidos y con el número de rutas no óptimas?
2. ¿A qué estrategia se parece A\* cuando el factor es muy grande? ¿Por qué?

In [ ]:
for factor in [1, 1.5, 2, 4]:
    exp_total, malas = 0, 0
    for o in ciudades:
        for d in ciudades:
            if o == d:
                continue
            optimo, _ = resolver_ruta(o, d, "coste uniforme")
            nodo, e = resolver_ruta(o, d, "A*", factor=factor)
            exp_total += e
            malas += nodo.g > optimo.g
    print(f"factor {factor:>3}: expandidos promedio {exp_total / n:.2f} | rutas no óptimas: {malas}")

*Tus respuestas al ejercicio 2:*

## 3. El 8-puzle: dos heurísticas, dos resultados

En el mapa de España el ahorro es pequeño porque el espacio es diminuto. En el **8-puzle** el espacio tiene 181.440 estados alcanzables, y ahí la heurística marca la diferencia. Usamos las dos heurísticas clásicas de Russell y Norvig (2004, cap. 4):

- $h_1$: número de fichas **mal colocadas**.
- $h_2$: suma de las **distancias de Manhattan** de cada ficha a su posición objetivo.

El estado es una tupla de 9 números leída por filas; el 0 es el hueco.

In [ ]:
OBJETIVO_8 = (1, 2, 3, 4, 5, 6, 7, 8, 0)
POS_OBJ = {v: divmod(i, 3) for i, v in enumerate(OBJETIVO_8)}


def sucesores_8(estado):
    i = estado.index(0)
    f, c = divmod(i, 3)
    res = []
    for df, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nf, nc = f + df, c + dc
        if 0 <= nf < 3 and 0 <= nc < 3:
            j = nf * 3 + nc
            e = list(estado)
            e[i], e[j] = e[j], e[i]
            res.append((tuple(e), 1))
    return res


def h0(e):
    return 0                                       # sin información: coste uniforme


def h1(e):
    return sum(1 for i, v in enumerate(e) if v != 0 and v != OBJETIVO_8[i])


def h2(e):
    total = 0
    for i, v in enumerate(e):
        if v:
            f, c = divmod(i, 3)
            fo, co = POS_OBJ[v]
            total += abs(f - fo) + abs(c - co)
    return total


def a_estrella_8(inicial, h):
    return primero_el_mejor(inicial, lambda e: e == OBJETIVO_8, sucesores_8,
                            lambda n: n.g + h(n.estado))


def mostrar_8(e):
    for f in range(3):
        print(" ".join(str(v) if v else "·" for v in e[f * 3:f * 3 + 3]))


inicial = (7, 2, 4, 5, 0, 6, 8, 3, 1)               # un estado inicial difícil
mostrar_8(inicial)
print("h1 =", h1(inicial), "| h2 =", h2(inicial))
for nombre, h in [("h0 (coste uniforme)", h0), ("h1", h1), ("h2", h2)]:
    t0 = time.perf_counter()
    nodo, exp = a_estrella_8(inicial, h)
    print(f"{nombre:<20} pasos: {nodo.g:>3}  expandidos: {exp:>7}  tiempo: {time.perf_counter() - t0:.2f} s")

### ✏️ Ejercicio 3. El factor de ramificación efectivo

El **factor de ramificación efectivo** $b^*$ resume en un número cuánto «poda» una heurística: es el factor de ramificación que tendría un árbol uniforme de profundidad $d$ con $N + 1$ nodos, $N + 1 = 1 + b^* + (b^*)^2 + \dots + (b^*)^d$ (Russell y Norvig, 2004, cap. 4). Cuanto más cerca de 1, mejor.

La celda genera 15 puzles aleatorios para cada profundidad de solución y promedia los nodos expandidos y $b^*$ (puede tardar alrededor de un minuto).

1. Compara la tabla con la de Russell y Norvig (2004, cap. 4) para el 8-puzle. ¿Se parecen las tendencias?
2. $h_2$ **domina** a $h_1$: para todo estado, $h_2 \geq h_1$. ¿Por qué eso implica que A\* con $h_2$ nunca expande más nodos que con $h_1$ (sección 3.1.3)?
3. ¿Por qué no se incluye coste uniforme en profundidades altas?

In [ ]:
def b_efectivo(N, d):
    # Resuelve N + 1 = 1 + b + b^2 + ... + b^d por bisección.
    if d == 0:
        return float("nan")
    lo, hi = 1.0, 10.0
    for _ in range(60):
        b = (lo + hi) / 2
        total = sum(b ** k for k in range(d + 1))
        lo, hi = (b, hi) if total < N + 1 else (lo, b)
    return (lo + hi) / 2


def puzle_aleatorio(pasos, rng):
    e, previo = OBJETIVO_8, None
    for _ in range(pasos):
        opciones = [s for s, _ in sucesores_8(e) if s != previo]
        previo, e = e, rng.choice(opciones)
    return e


rng = random.Random(42)
por_d = {}
while any(len(por_d.get(d, [])) < 15 for d in (4, 8, 12, 16, 20)):
    e = puzle_aleatorio(rng.randint(4, 60), rng)
    d = a_estrella_8(e, h2)[0].g
    if d in (4, 8, 12, 16, 20) and len(por_d.setdefault(d, [])) < 15:
        por_d[d].append(e)

print(f"{'d':>3} | {'N coste unif.':>13} {'N h1':>8} {'N h2':>7} | {'b* CU':>6} {'b* h1':>6} {'b* h2':>6}")
for d in sorted(por_d):
    fila = []
    for h in (h0, h1, h2):
        if h is h0 and d > 12:
            fila.append(None)
            continue
        fila.append(sum(a_estrella_8(e, h)[1] for e in por_d[d]) / len(por_d[d]))
    txt_n = " ".join(f"{x:>{w}.0f}" if x is not None else f"{'—':>{w}}" for x, w in zip(fila, (13, 8, 7)))
    txt_b = " ".join(f"{b_efectivo(x, d):>6.2f}" if x is not None else f"{'—':>6}" for x in fila)
    print(f"{d:>3} | {txt_n} | {txt_b}")

*Tus respuestas al ejercicio 3:*

## 4. Vuelta atrás: el problema de las *N* reinas

El problema consiste en colocar *N* reinas en un tablero de *N* × *N* sin que ninguna ataque a otra. La **vuelta atrás** (*backtracking*) coloca una reina por columna y, en cuanto una colocación viola una restricción, **deshace** el último paso y prueba la siguiente opción (sección 3.2). Así poda ramas enteras del árbol sin generarlas.

Comparamos con la fuerza bruta, que genera todas las permutaciones (una reina por fila y columna) y solo después comprueba las diagonales.

In [ ]:
from itertools import permutations


def reinas_backtracking(n):
    # Devuelve (primera solución, nodos generados).
    columnas = []                                   # columnas[i] = fila de la reina de la columna i
    generados = 0

    def es_valida(fila):
        c = len(columnas)
        return all(f != fila and abs(f - fila) != c - i for i, f in enumerate(columnas))

    def colocar():
        nonlocal generados
        if len(columnas) == n:
            return True
        for fila in range(n):
            generados += 1
            if es_valida(fila):
                columnas.append(fila)
                if colocar():
                    return True
                columnas.pop()                      # vuelta atrás
        return False

    return (list(columnas) if colocar() else None), generados


def reinas_fuerza_bruta(n):
    for k, p in enumerate(permutations(range(n)), 1):
        if all(abs(p[i] - p[j]) != j - i for i in range(n) for j in range(i + 1, n)):
            return list(p), k
    return None, k


def dibujar_reinas(sol):
    n = len(sol)
    for fila in range(n):
        print(" ".join("♛" if sol[c] == fila else "·" for c in range(n)))


sol, gen = reinas_backtracking(8)
dibujar_reinas(sol)
print("\nNodos generados con vuelta atrás:", gen)

### ✏️ Ejercicio 4. ¿Cuánto poda la vuelta atrás?

1. Ejecuta la celda. ¿Cómo crecen los nodos generados por la vuelta atrás y las permutaciones revisadas por la fuerza bruta al aumentar *N*?
2. La fuerza bruta ya usa una formulación «mejorada» (una reina por fila y columna). ¿Qué hace la vuelta atrás que la fuerza bruta no hace?
3. Observa que el número de nodos no crece de forma regular con *N* (por ejemplo, compara *N* = 6 y *N* = 7). ¿A qué crees que se debe?

In [ ]:
print(f"{'N':>3} {'vuelta atrás':>14} {'fuerza bruta':>14}")
for n in range(4, 11):
    _, gb = reinas_backtracking(n)
    _, gf = reinas_fuerza_bruta(n) if n <= 9 else (None, None)
    print(f"{n:>3} {gb:>14} {gf if gf is not None else '—':>14}")

*Tus respuestas al ejercicio 4:*

---
# Parte II. Optimización: el viajante de comercio por Colombia

## 5. El problema

Una empresa quiere que un representante visite las **24 ciudades** de la lista, empezando y terminando en Bogotá, y recorriendo la menor distancia posible. Es el problema del viajante de comercio (TSP) de la sección 2.2.1. Con 24 ciudades hay $23!/2 \approx 1{,}3 \times 10^{22}$ recorridos posibles: ni el computador más rápido puede revisarlos todos.

Para simplificar usamos la **distancia en línea recta** entre las coordenadas aproximadas de cada ciudad. Las distancias reales por carretera son mayores, pero el problema conserva su dificultad.

Aquí ya no buscamos un **camino** de acciones, sino una **configuración** (un recorrido) que minimice una función objetivo. Por eso usamos **búsqueda local**: partimos de un recorrido completo y lo mejoramos con pequeños cambios (sección 3.5).

In [ ]:
capitales = {
    "Bogotá": (4.711, -74.072),       "Medellín": (6.244, -75.581),
    "Cali": (3.452, -76.532),         "Barranquilla": (10.964, -74.796),
    "Cartagena": (10.391, -75.479),   "Bucaramanga": (7.119, -73.123),
    "Cúcuta": (7.894, -72.508),       "Pereira": (4.813, -75.696),
    "Manizales": (5.070, -75.514),    "Armenia": (4.534, -75.681),
    "Ibagué": (4.438, -75.232),       "Neiva": (2.927, -75.282),
    "Pasto": (1.214, -77.281),        "Popayán": (2.444, -76.614),
    "Villavicencio": (4.142, -73.627), "Tunja": (5.535, -73.367),
    "Montería": (8.748, -75.881),     "Sincelejo": (9.304, -75.398),
    "Valledupar": (10.463, -73.253),  "Santa Marta": (11.241, -74.205),
    "Riohacha": (11.544, -72.907),    "Quibdó": (5.694, -76.658),
    "Yopal": (5.337, -72.395),        "Florencia": (1.614, -75.606),
}
nombres = list(capitales)                       # nombres[0] = Bogotá, punto de partida
D_colombia = [[linea_recta(a, b, capitales) for b in nombres] for a in nombres]


def usar_instancia(matriz):
    # Todas las funciones de esta parte trabajan sobre la instancia activa (D, N).
    global D, N
    D, N = matriz, len(matriz)


usar_instancia(D_colombia)


def longitud(ruta):
    # ruta: permutación de 0..N-1 que empieza en 0; se vuelve al inicio al final.
    return sum(D[ruta[i]][ruta[(i + 1) % N]] for i in range(N))


def dibujar(ruta, titulo):
    xs = [capitales[nombres[i]][1] for i in ruta + [ruta[0]]]
    ys = [capitales[nombres[i]][0] for i in ruta + [ruta[0]]]
    plt.figure(figsize=(4.5, 5.5))
    plt.plot(xs, ys, "-o", ms=4)
    plt.plot(xs[0], ys[0], "rs", ms=8)
    for i in ruta:
        plt.annotate(nombres[i], (capitales[nombres[i]][1], capitales[nombres[i]][0]), fontsize=7)
    plt.title(f"{titulo}: {longitud(ruta):,.0f} km")
    plt.xlabel("longitud")
    plt.ylabel("latitud")
    plt.show()


rng = random.Random(7)
aleatoria = [0] + rng.sample(range(1, N), N - 1)
dibujar(aleatoria, "Recorrido aleatorio")

## 6. Un algoritmo constructivo voraz: el vecino más cercano

Antes de la búsqueda local, una idea simple: desde la ciudad actual, ir siempre a la **ciudad no visitada más cercana**. Es un algoritmo **constructivo voraz** (sección 3.4): construye la solución paso a paso tomando la mejor decisión local, sin volver atrás.

In [ ]:
def vecino_mas_cercano(inicio=0):
    ruta, pendientes = [inicio], set(range(N)) - {inicio}
    while pendientes:
        actual = ruta[-1]
        siguiente = min(pendientes, key=lambda j: D[actual][j])
        ruta.append(siguiente)
        pendientes.remove(siguiente)
    return ruta


voraz = vecino_mas_cercano()
dibujar(voraz, "Vecino más cercano")

## 7. *Hill climbing* con vecindario 2-opt

Un **movimiento 2-opt** elimina dos aristas del recorrido y reconecta los tramos invirtiendo el segmento intermedio. Si el recorrido se cruza consigo mismo, un 2-opt deshace el cruce. Todos los recorridos que se obtienen con un 2-opt forman el **vecindario** del recorrido actual.

El *hill climbing* (ascensión de colinas; aquí, como minimizamos, «descenso») pasa al **mejor vecino** mientras alguno mejore el recorrido, y se detiene en un **óptimo local** (sección 3.5).

In [ ]:
def delta_2opt(ruta, i, j):
    # Cambio de longitud al invertir ruta[i..j] (1 <= i < j <= N-1).
    a, b = ruta[i - 1], ruta[i]
    c, d = ruta[j], ruta[(j + 1) % N]
    return D[a][c] + D[b][d] - D[a][b] - D[c][d]


def aplicar_2opt(ruta, i, j):
    return ruta[:i] + ruta[i:j + 1][::-1] + ruta[j + 1:]


def hill_climbing(ruta):
    historia = [longitud(ruta)]
    while True:
        mejor, mov = 0, None
        for i in range(1, N - 1):
            for j in range(i + 1, N):
                d = delta_2opt(ruta, i, j)
                if d < mejor - 1e-9:
                    mejor, mov = d, (i, j)
        if mov is None:
            return ruta, historia                  # óptimo local: ningún vecino mejora
        ruta = aplicar_2opt(ruta, *mov)
        historia.append(longitud(ruta))


hc, hist_hc = hill_climbing(aleatoria)
print(f"Desde el recorrido aleatorio: {hist_hc[0]:,.0f} km -> {hist_hc[-1]:,.0f} km en {len(hist_hc) - 1} pasos")
dibujar(hc, "Hill climbing (2-opt)")

### ✏️ Ejercicio 5. Óptimos locales y reinicios

1. La celda siguiente ejecuta el *hill climbing* desde 30 recorridos aleatorios distintos. ¿Llegan todos al mismo resultado? ¿Qué te dice eso sobre los **óptimos locales**?
2. El **reinicio aleatorio** consiste en quedarse con el mejor de varios *hill climbing*. ¿Cuál fue el mejor resultado de los 30? ¿Cuántas ejecuciones lo alcanzaron?
3. ¿Mejora el *hill climbing* si parte del recorrido del vecino más cercano en lugar de uno aleatorio?

In [ ]:
resultados_hc = []
for semilla in range(30):
    r = random.Random(semilla)
    inicio = [0] + r.sample(range(1, N), N - 1)
    resultados_hc.append(longitud(hill_climbing(inicio)[0]))

print("Mínimo:", f"{min(resultados_hc):,.0f} km", "| Máximo:", f"{max(resultados_hc):,.0f} km",
      "| Promedio:", f"{sum(resultados_hc) / len(resultados_hc):,.0f} km")
print("Ejecuciones que alcanzaron el mínimo:", sum(abs(x - min(resultados_hc)) < 1 for x in resultados_hc))
print("Desde el vecino más cercano:", f"{longitud(hill_climbing(voraz)[0]):,.0f} km")

*Tus respuestas al ejercicio 5:*

## 8. Temple simulado

El **temple simulado** (*simulated annealing*) elige un vecino **al azar**. Si es mejor, lo acepta; si es peor en $\Delta$ km, lo acepta con probabilidad $e^{-\Delta / T}$. La **temperatura** $T$ empieza alta (se aceptan muchos empeoramientos y se explora) y se reduce poco a poco (se aceptan cada vez menos y se explota) (Kirkpatrick et al., 1983; sección 3.6).

In [ ]:
def temple_simulado(ruta, T0=1000.0, alfa=0.995, T_min=0.1, iter_por_T=100, semilla=0):
    rng = random.Random(semilla)
    actual, L = list(ruta), longitud(ruta)
    mejor, L_mejor = list(actual), L
    historia, T = [], T0
    while T > T_min:
        for _ in range(iter_por_T):
            i, j = sorted(rng.sample(range(1, N), 2))
            d = delta_2opt(actual, i, j)
            if d < 0 or rng.random() < math.exp(-d / T):
                actual = aplicar_2opt(actual, i, j)
                L += d
                if L < L_mejor:
                    mejor, L_mejor = list(actual), L
        historia.append(L)
        T *= alfa                                   # enfriamiento geométrico
    return mejor, historia


t0 = time.perf_counter()
ts, hist_ts = temple_simulado(aleatoria)
print(f"Temple simulado: {longitud(ts):,.0f} km en {time.perf_counter() - t0:.1f} s")
plt.figure(figsize=(6, 3))
plt.plot(hist_ts)
plt.xlabel("Paso de temperatura")
plt.ylabel("Longitud actual (km)")
plt.title("Evolución del temple simulado")
plt.show()

### ✏️ Ejercicio 6. La temperatura importa

1. Observa la gráfica. ¿Por qué la longitud **sube y baja** al principio y se estabiliza al final?
2. Ejecuta la celda siguiente con temperaturas iniciales de 1, 100, 1.000 y 10.000. ¿Qué pasa con $T_0 = 1$? ¿A qué algoritmo se parece? ¿Y con una temperatura muy alta?
3. Prueba un enfriamiento más rápido (`alfa=0.9`). ¿Qué observas?

In [ ]:
for T0 in [1, 100, 1000, 10000]:
    longitudes = [longitud(temple_simulado(aleatoria, T0=T0, semilla=s)[0]) for s in range(5)]
    print(f"T0 = {T0:>6}: mejor {min(longitudes):,.0f} km | promedio {sum(longitudes) / 5:,.0f} km")

*Tus respuestas al ejercicio 6:*

## 9. Búsqueda tabú

La **búsqueda tabú** se mueve siempre al **mejor vecino**, aunque empeore, y para no regresar a donde estaba mantiene una **lista tabú** con los movimientos recientes, que quedan prohibidos durante unas iteraciones (la **permanencia** o *tenure*). Un movimiento tabú se permite si produce un recorrido mejor que el mejor conocido: es el **criterio de aspiración** (Glover, 1986; sección 3.7).

Aquí el atributo tabú son las dos aristas que se acaban de crear: durante `tenure` iteraciones no se pueden eliminar. Si en algún momento todos los movimientos fueran tabú, la lista se vacía.

In [ ]:
def busqueda_tabu(ruta, iteraciones=300, tenure=6):
    actual = list(ruta)
    L = longitud(actual)
    mejor, L_mejor = list(actual), L
    tabu = {}                                       # arista -> iteración hasta la que es tabú
    historia = []
    for it in range(iteraciones):
        candidato = None
        for i in range(1, N - 1):
            for j in range(i + 1, N):
                d = delta_2opt(actual, i, j)
                a, b = actual[i - 1], actual[i]
                c, e = actual[j], actual[(j + 1) % N]
                es_tabu = tabu.get(frozenset((a, b)), -1) >= it or tabu.get(frozenset((c, e)), -1) >= it
                aspiracion = L + d < L_mejor - 1e-9
                if (not es_tabu or aspiracion) and (candidato is None or d < candidato[0]):
                    candidato = (d, i, j, a, c, b, e)
        if candidato is None:                       # todos los movimientos son tabú
            tabu.clear()
            continue
        d, i, j, a, c, b, e = candidato
        actual = aplicar_2opt(actual, i, j)
        L += d
        tabu[frozenset((a, c))] = it + tenure       # no deshacer las aristas recién creadas
        tabu[frozenset((b, e))] = it + tenure
        if L < L_mejor - 1e-9:
            mejor, L_mejor = list(actual), L
        historia.append(L)
    return mejor, historia


t0 = time.perf_counter()
tb, hist_tb = busqueda_tabu(aleatoria)
print(f"Búsqueda tabú: {longitud(tb):,.0f} km en {time.perf_counter() - t0:.1f} s")
plt.figure(figsize=(6, 3))
plt.plot(hist_tb)
plt.xlabel("Iteración")
plt.ylabel("Longitud actual (km)")
plt.title("Evolución de la búsqueda tabú")
plt.show()

Observa que, a diferencia del *hill climbing*, la búsqueda tabú **no se detiene** en el primer óptimo local: la curva vuelve a subir cuando el algoritmo acepta un movimiento que empeora para escapar de él.

## 10. Algoritmo genético

Un **algoritmo genético** mantiene una **población** de recorridos. En cada generación selecciona a los mejores por **torneo**, los **cruza** para crear hijos que combinan partes de ambos padres y los **muta** con una pequeña probabilidad (Holland, 1975; sección 3.8).

Para permutaciones no sirve el cruce clásico de un punto, porque produciría recorridos con ciudades repetidas. Usamos el **cruce de orden** (*order crossover*, OX): el hijo copia un segmento del primer padre y completa el resto con las ciudades que faltan, en el orden en que aparecen en el segundo padre.

In [ ]:
def cruce_ox(p1, p2, rng):
    # Cruce de orden sobre las posiciones 1..N-1 (la posición 0 es siempre Bogotá).
    i, j = sorted(rng.sample(range(1, N), 2))
    hijo = [None] * N
    hijo[0] = 0
    hijo[i:j + 1] = p1[i:j + 1]
    usados = set(hijo[i:j + 1]) | {0}
    resto = [c for c in p2[1:] if c not in usados]
    k = 0
    for pos in range(1, N):
        if hijo[pos] is None:
            hijo[pos] = resto[k]
            k += 1
    return hijo


def mutar(ruta, prob, rng):
    if rng.random() < prob:
        i, j = sorted(rng.sample(range(1, N), 2))
        ruta = aplicar_2opt(ruta, i, j)             # mutación por inversión de un segmento
    return ruta


def genetico(tam_pob=100, generaciones=400, prob_mut=0.3, torneo=3, elite=2, semilla=0):
    rng = random.Random(semilla)
    poblacion = [[0] + rng.sample(range(1, N), N - 1) for _ in range(tam_pob)]
    historia = []
    for _ in range(generaciones):
        poblacion.sort(key=longitud)
        historia.append(longitud(poblacion[0]))
        nueva = [list(r) for r in poblacion[:elite]]          # elitismo
        while len(nueva) < tam_pob:
            p1 = min(rng.sample(poblacion, torneo), key=longitud)
            p2 = min(rng.sample(poblacion, torneo), key=longitud)
            nueva.append(mutar(cruce_ox(p1, p2, rng), prob_mut, rng))
        poblacion = nueva
    mejor = min(poblacion, key=longitud)
    return mejor, historia


t0 = time.perf_counter()
ag, hist_ag = genetico()
print(f"Algoritmo genético: {longitud(ag):,.0f} km en {time.perf_counter() - t0:.1f} s")
plt.figure(figsize=(6, 3))
plt.plot(hist_ag)
plt.xlabel("Generación")
plt.ylabel("Mejor longitud (km)")
plt.title("Evolución del algoritmo genético")
plt.show()

## 11. Comparación final

Con 24 ciudades casi todos los métodos llegan al mismo recorrido: la instancia es pequeña para ellos. Para ver diferencias, la comparación se hace también sobre una instancia **más difícil**: 100 puntos aleatorios en un cuadrado de 1.000 × 1.000 km, generados con una semilla fija para que todos obtengamos la misma instancia.

La celda ejecuta cada método con 5 semillas distintas sobre las dos instancias (puede tardar unos minutos) y resume la mejor longitud, el promedio y el tiempo promedio. Los métodos con azar pueden dar resultados distintos en cada ejecución: por eso se comparan con varias semillas.

In [ ]:
def correr(nombre, funcion, semillas=range(5)):
    longs, tiempos = [], []
    for s in semillas:
        t0 = time.perf_counter()
        ruta = funcion(s)
        tiempos.append(time.perf_counter() - t0)
        longs.append(longitud(ruta))
    return nombre, min(longs), sum(longs) / len(longs), sum(tiempos) / len(tiempos)


def inicio_aleatorio(s):
    r = random.Random(100 + s)
    return [0] + r.sample(range(1, N), N - 1)


metodos = [
    ("Vecino más cercano", lambda s: vecino_mas_cercano()),
    ("Hill climbing", lambda s: hill_climbing(inicio_aleatorio(s))[0]),
    ("Temple simulado", lambda s: temple_simulado(inicio_aleatorio(s), semilla=s)[0]),
    ("Búsqueda tabú", lambda s: busqueda_tabu(inicio_aleatorio(s))[0]),
    ("Algoritmo genético", lambda s: genetico(semilla=s)[0]),
]
rp = random.Random(2026)
puntos = [(rp.uniform(0, 1000), rp.uniform(0, 1000)) for _ in range(100)]
D_aleatoria = [[math.dist(p, q) for q in puntos] for p in puntos]

for titulo, matriz in [("24 capitales de Colombia", D_colombia), ("100 puntos aleatorios", D_aleatoria)]:
    usar_instancia(matriz)
    print(f"\n{titulo}")
    print(f"{'Método':<20} {'Mejor (km)':>11} {'Promedio (km)':>14} {'Tiempo (s)':>11}")
    for nombre, mejor, prom, t in [correr(n, f) for n, f in metodos]:
        print(f"{nombre:<20} {mejor:>11,.0f} {prom:>14,.0f} {t:>11.2f}")

usar_instancia(D_colombia)

In [ ]:
mejor_global = min([ts, tb, ag, hc], key=longitud)
dibujar(mejor_global, "Mejor recorrido encontrado")

### ✏️ Ejercicio 7. ¿Qué método elegirías?

1. Con las dos tablas anteriores, ordena los métodos según la calidad de la solución y según el tiempo. ¿El orden es el mismo en las dos instancias? ¿Hay un ganador claro?
2. Ninguno de estos métodos **garantiza** el óptimo. ¿Por qué aun así son útiles? Relaciona tu respuesta con el tamaño del espacio de búsqueda.
3. En la instancia de 100 puntos el algoritmo genético queda incluso por detrás del vecino más cercano. Sobre esa instancia (`usar_instancia(D_aleatoria)`), aumenta `generaciones` y cambia `prob_mut` y `tam_pob`. ¿Cuánto mejora y cuánto tiempo cuesta? Haz lo mismo con el `tenure` de la búsqueda tabú (prueba valores entre 2 y 10).
4. Si tuvieras que integrar uno de ellos en una aplicación de reparto a domicilio que debe responder en menos de un segundo, ¿cuál usarías y con qué ajustes?

*Tus respuestas al ejercicio 7:*

## 12. Cierre

Completa esta tabla con lo que observaste:

| Técnica | Tipo (camino / optimización) | ¿Garantiza el óptimo? | Ventaja principal | Desventaja principal |
|---|---|---|---|---|
| Voraz primero el mejor | | | | |
| A\* | | | | |
| Vuelta atrás | | | | |
| Vecino más cercano | | | | |
| *Hill climbing* | | | | |
| Temple simulado | | | | |
| Búsqueda tabú | | | | |
| Algoritmo genético | | | | |

Lleva el cuaderno ejecutado a la sesión sincrónica. Las técnicas de esta práctica son la base de la primera parte de la **Actividad Evaluativa 2**.

---

**Referencias**

García Serrano, A. (2016). *Inteligencia artificial: Fundamentos, práctica y aplicaciones* (2.ª ed.). RC Libros.

Glover, F. (1986). Future paths for integer programming and links to artificial intelligence. *Computers & Operations Research, 13*(5), 533-549. https://doi.org/10.1016/0305-0548(86)90048-1

Holland, J. H. (1975). *Adaptation in natural and artificial systems*. University of Michigan Press.

Kirkpatrick, S., Gelatt, C. D., y Vecchi, M. P. (1983). Optimization by simulated annealing. *Science, 220*(4598), 671-680. https://doi.org/10.1126/science.220.4598.671

Russell, S. J., y Norvig, P. (2004). *Inteligencia artificial: Un enfoque moderno* (2.ª ed.; J. M. Corchado Rodríguez et al., Trads.). Pearson Educación.